# 08 — Component Detection with YOLOv8

**Question:** can a small YOLOv8n fine-tuned on ~500 hand-labelled teardown images find the major internal components (battery, mainboard, camera, display, speaker, connector) well enough for the demo?

**Target:** mAP@0.5 ≥ 0.50 — deliberately modest given the tiny dataset.

## Setup

In [ ]:
from pathlib import Path
import os, yaml
DATA_DIR = Path('../data/raw/teardowns')  # images/, labels/, train.txt, val.txt
ART = Path('../ml/artifacts'); ART.mkdir(exist_ok=True, parents=True)
CLASSES = ['battery','mainboard','camera','display','speaker','connector']

## Write dataset YAML

In [ ]:
cfg = {'path': str(DATA_DIR.resolve()), 'train': 'train.txt', 'val': 'val.txt', 'names': {i:c for i,c in enumerate(CLASSES)}}
yaml_path = DATA_DIR/'teardowns.yaml'
yaml_path.write_text(yaml.safe_dump(cfg))
yaml_path

## Fine-tune

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(data=str(yaml_path), epochs=40, imgsz=640, batch=16, seed=42, project=str(ART), name='detector')

## Evaluate

In [ ]:
metrics = model.val()
print({'mAP50': metrics.box.map50, 'mAP50-95': metrics.box.map})

## Export

In [ ]:
model.save(str(ART/'detector.pt'))
import json
(ART/'detector_metrics.json').write_text(json.dumps({'mAP50': float(metrics.box.map50), 'mAP50-95': float(metrics.box.map), 'classes': CLASSES}, indent=2))